# Context-specific, directional, genetics-anchored drug-target nomination
### Genome-scale CD4⁺ T-cell CRISPRi Perturb-seq → ranked drug targets

Reproducible pipeline for the analysis in `findings_report.md`.

**Data**: Zhu, Dann, … Pritchard & Marson (2025), bioRxiv 10.64898/2025.12.23.696273 —
`GWCD4i.DE_stats.h5ad` (per-perturbation DE statistics, 3 conditions: Rest / Stim 8h / Stim 48h).

**Pipeline**: (1) build pathogenic pro-inflammatory program signature → (2) directional
context-specific scoring with empirical null → (3) human-genetics support (Open Targets +
gnomAD) → (4) druggability/tractability → (5) clinical novelty → (6) integrated ranking →
(7) deep-dive vignettes.

> All heavy intermediates are checkpointed as artifacts; each step below can be run from the
> upstream checkpoint without re-streaming the 16.8 GB source file.

## 0. Environment

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mp
pd.set_option('display.width', 160)
# Analysis was run in the `python` env (numpy/pandas/scipy/matplotlib) + `figure-style` skill.

## 1. Acquire & validate inputs

The source `.h5ad` is 16.8 GB — larger than local disk. We streamed it from S3
(`s3://genome-scale-tcell-perturb-seq/marson2025_data/`) and extracted only what the
directional score needs: the readout z-score matrix plus obs/var metadata. Those extracts
are saved as artifacts:

- `de_obs.parquet` — 33,983 rows (perturbation × condition); cols include
  `target_gene, condition, target_contrast (=Ensembl ID), n_downstream, ontarget_effect_size`.
- `de_var.parquet` — 10,282 readout genes (`gene_name`).
- `zscore_f32.npy` — 33,983 × 10,282 float32 readout z-scores (memory-mapped).

In [ ]:
de_obs = pd.read_parquet('de_obs.parquet')
de_var = pd.read_parquet('de_var.parquet')
print('obs', de_obs.shape, '| var', de_var.shape)
print(de_obs[['target_gene','condition','ontarget_effect_size','n_downstream']].head())
# Z = np.load('inputs/zscore_f32.npy', mmap_mode='r')   # (33983, 10282)

## 2. Pathogenic pro-inflammatory program signature

Signed gene set: 36 pro-inflammatory (+1) and 12 regulatory (−1). 48/50 measured
(*IL17A, RORA* absent from readout). Program score for a perturbation = mean over signature
genes of (signed weight × readout z-score).

In [ ]:
sig = pd.read_csv('program_signature.csv')   # columns: gene, weight (+1 / -1)
print(sig['weight'].value_counts())
print('signature genes measured:', len(sig))

*Figure: `program_signature_heatmap.png` — signature readout across positive-control perturbations.*

## 3. Directional, context-specific scoring + empirical null

Per perturbation × condition: program score; then 500 random signed gene sets (matched
36↑/12↓, drawn from non-signature genes) → empirical z (`emp_z`) and BH-FDR per condition.

- **direction**: `driver_antagonize` (KD lowers program) vs `brake_agonize` (KD raises it)
- **context_specificity**: 0 = uniform across conditions, 1 = single-condition
- **quality gate**: `peak_kd < -1.0` AND `peak_emp_fdr < 0.05` AND `|peak score| > 0.2`

In [ ]:
scores = pd.read_parquet('perturbation_scores_wide.parquet')
print('perturbations scored:', len(scores))

def program_score(Zc, sig_idx, sign):
    return (Zc[:, sig_idx] * sign).mean(axis=1)

def empirical_null(Zc, sig_idx, sign, n_draws=500, seed=0):
    rng = np.random.default_rng(seed)
    nonsig = np.setdiff1d(np.arange(Zc.shape[1]), sig_idx)
    null = np.empty((n_draws, Zc.shape[0]))
    for d in range(n_draws):
        pick = rng.choice(nonsig, size=len(sig_idx), replace=False)
        null[d] = (Zc[:, pick] * sign).mean(axis=1)
    return null.mean(0), null.std(0)
# emp_z = (observed - null_mean) / null_std ; then BH-FDR within each condition

*Figure: `perturbation_directionality.png`.*

## 4. Human-genetics support (Open Targets + gnomAD)

Each candidate's Ensembl ID queried via the `clinical-genomics` MCP (Open Targets GraphQL)
for autoimmune/allergic associations, LoF-constraint, tractability, and known drugs.
Genetics tier ∈ {strong, moderate, weak, none}.

In [ ]:
OT_QUERY = '''query($ensg:String!){ target(ensemblId:$ensg){
  approvedSymbol
  geneticConstraint{ constraintType score upperBin }
  tractability{ modality label value }
  associatedDiseases(page:{size:8,index:0}){ count rows{ score
     disease{ id name therapeuticAreas{ id name } } } }
  drugAndClinicalCandidates{ count rows{ maxClinicalStage
     drug{ name drugType maximumClinicalStage } } }
}}'''
# In the repl tool:
# host.mcp('clinical-genomics','open_targets_graphql', query=OT_QUERY, variables={'ensg': ensembl_id})
genetics = pd.read_parquet('ot_annotation.parquet')   # checkpoint
print(genetics.shape)

*Figure: `genetics_support.png`.*

## 5. Druggability & tractability

Modality priority (respecting biology): secreted cytokine → biologic; surface
cytokine/catalytic receptor → antibody; kinase/enzyme/GPCR/ion-channel/NR/transporter →
small molecule; else SM-ligandable → small molecule; surface antigen → antibody; TF w/o
pocket → hard (degrader/PPI).

In [ ]:
cand5 = pd.read_parquet('candidates_step5.parquet')   # checkpoint
print(cand5['suggested_modality'].value_counts())

*Figure: `druggability_landscape.png`.*

## 6. Clinical novelty

Open Targets known-drug / clinical-candidate max stage → {novel_undrugged,
preclinical_or_unknown, clinical_stage_drug, approved_drug_exists}.

In [ ]:
cand6 = pd.read_parquet('candidates_step6.parquet')   # checkpoint
print(cand6['novelty_class'].value_counts())

*Figure: `novelty_breakdown.png`.*

## 7. Integrated ranking

Four components in [0,1], weighted sum. Directionality kept as annotation.

| Component | Weight |
|---|---|
| Causal strength | 0.34 |
| Human genetics | 0.30 |
| Druggability | 0.22 |
| Clinical novelty | 0.14 |

In [ ]:
df = pd.read_parquet('candidates_step7.parquet')   # final scored table
W = dict(causal=0.34, genetics=0.30, drug=0.22, novelty=0.14)
# df['integrated_score'] = (W['causal']*df['causal_component'] + W['genetics']*df['genetics_component']
#                           + W['drug']*df['drug_component'] + W['novelty']*df['novelty_component'])
shortlist = df.sort_values('integrated_score', ascending=False)
print('POSITIVE CONTROLS (should rank high):')
for g in ['IL4R','IL2RA','STAT3','IL6R','TBX21','BATF']:
    if g in df.index:
        r = df.loc[g]; print(f'  {g:6s} rank {int(r["rank"]):4d}  score {r["integrated_score"]:.2f}  {r["direction"]}')
print('\nnovel-priority nominations:', int(df['novel_priority'].sum()))
shortlist.head(15)[['rank','integrated_score','direction','genetics_tier','suggested_modality','novelty_class']]

*Figure: `top_targets_evidence_matrix.png`. Output CSV: `target_shortlist.csv`.*

## 8. Deep-dive: top novel nominations

Six novel-priority targets across both directions and diverse immune diseases.

In [ ]:
picks = ['STAT5B','CTSH','NEK6','PTPN2','SIK2','STAT6']
vig = pd.read_csv('deep_dive_vignettes.csv', index_col=0)
vig[['rank','integrated_score','direction','peak_condition','genetics_autoimmune',
     'ot_autoimmune_disease','suggested_modality']]

*Figure: `top_target_vignettes.png` — context-specific program-score trajectories with empirical-null significance.*

## Limitations

- Computational hypotheses; **no wet-lab validation**.
- Program score depends on the 48-gene signature (*IL17A/RORA* unmeasured → Th17 under-represented).
- Open Targets genetics mixes common-variant and Mendelian evidence; tiers don't encode direction of genetic effect.
- Modality assignment is rule-based from protein class.
- Findings specific to CD4⁺ T cells under CD3/CD28-type stimulation.